In [12]:
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.embeddings import DashScopeEmbeddings

from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [23]:
embedding = DashScopeEmbeddings(model="text-embedding-v2", dashscope_api_key="sk-f22c4fa77bab4a42a47486922c84a467")
# reranker = DashScopeRerank(model="gte-rerank")
vector_store = Chroma(
    persist_directory="/Users/xiaohuxu/Documents/python/deepmodeling/OrcaMul/servers/multiwfn/database/vector_db_qwen",
    embedding_function=embedding
)

In [24]:
results = vector_store.similarity_search(
    "using shell script to run Multiwfn",
    k=2,
    # filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

# results = vector_store.similarity_search_by_vector(
#     embedding=embedding.embed_query("using shell script to run Multiwfn"), k=1
# )
# for doc in results:
#     print(f"* {doc.page_content} [{doc.metadata}]")

* 5  Skills 
 
1099 
Content orbana_1_3.in | ./Multiwfn.exe COCl2.fch > orbana_1_3.txt 
For Linux / MacOS users 
If you are a Linux or Mac OS user, you can not only run Multiwfn silently as introduced above, 
but also make use of "echo" command to avoid explicitly writing an input stream file. The last 
example can be equivalently realized by running this command: 
echo -e "8\n1\n1\n2\n3" | Multiwfn COCl2.fch > orbana_1_3.txt 
Each \n means pressing ENTER button once. 
If you prefer to use shell script, you can also add below lines into your shell script file: 
Multiwfn COCl2.fch > orbana_1_3.txt << EOF 
8 
1 
1 
2 
3 
EOF 
 
5.3 Running Multiwfn in batch mode 
Note: If you can read Chinese, please read Sections 3 and 4 of my blog article  “Detailed introduction to the 
command line running and batch running methods of Multiwfn ” (http://sobereva.com/612) instead, in which more 
detailed information about using shell script to run Multiwfn to automatically process batch of files is int

In [9]:
vector_store.__dir__()

['_client_settings',
 '_client',
 '_persist_directory',
 '_embedding_function',
 '_collection',
 'override_relevance_score_fn',
 '__module__',
 '__annotations__',
 '__doc__',
 '_LANGCHAIN_DEFAULT_COLLECTION_NAME',
 '__init__',
 'embeddings',
 '_Chroma__query_collection',
 'encode_image',
 'add_images',
 'add_texts',
 'similarity_search',
 'similarity_search_by_vector',
 'similarity_search_by_vector_with_relevance_scores',
 'similarity_search_with_score',
 '_select_relevance_score_fn',
 'similarity_search_by_image',
 'similarity_search_by_image_with_relevance_score',
 'max_marginal_relevance_search_by_vector',
 'max_marginal_relevance_search',
 'delete_collection',
 'get',
 'persist',
 'update_document',
 'update_documents',
 'from_texts',
 'from_documents',
 'delete',
 '__len__',
 '__abstractmethods__',
 '_abc_impl',
 'get_by_ids',
 'aget_by_ids',
 'adelete',
 'aadd_texts',
 'add_documents',
 'aadd_documents',
 'search',
 'asearch',
 '_euclidean_relevance_score_fn',
 '_cosine_relevance

In [11]:
vector_store._collection

Collection(name=langchain)

In [1]:
from langchain_community.document_loaders import DirectoryLoader

In [2]:
loader = DirectoryLoader("/Users/xiaohuxu/Documents/python/sob_blobs/sobereva_blogs_text", glob="**/*.md")
docs = loader.load()
len(docs)

114

In [3]:
print(docs[0].page_content[:100])

用于非限制性开壳层波函数的双正交化方法的原理与应用

URL: http://sobereva.com/448 ID: 448

用于非限制性开壳层波函数的双正交化方法的原理与应用

Author: 


In [7]:
docs[0].metadata

{'source': '/Users/xiaohuxu/Documents/python/sob_blobs/sobereva_blogs_text/448.md'}

In [8]:
docs[0].page_content

'用于非限制性开壳层波函数的双正交化方法的原理与应用\n\nURL: http://sobereva.com/448 ID: 448\n\n用于非限制性开壳层波函数的双正交化方法的原理与应用\n\nAuthor: sobereva\n\nDate: November 11, 2018\n\nCategory: Multiwfn\n\nViews: 21,478\n\n用于非限制性开壳层波函数的双正交化方法的原理与应用\n\nPrinciple and application of biorthogonalization method for unrestricted open-shell wavefunctions\n\n文/Sobereva@北京科音\n\nFirst release: 2018-Nov-11 Last update: 2021-Sep-10\n\n摘要 ：本文非常简要地介绍一下对非限制性开壳层波函数的alpha和beta轨道之间做双正交化的方法，并且以三重态乙醇为例介绍如何在Multiwfn中实现，使得其alpha和beta轨道最大程度匹配，从而便于讨论轨道。本文说的Multiwfn及其手册是官网上最新版本的情况。\n\n1 相关知识\n\n众所周知，非限制性开壳层计算（如UHF、UKS，以下简称为U）的时候alpha和beta是分别求解的，因此会产生alpha和beta这两套自旋轨道。对于自旋多重度>1的体系，以及自旋多重度为1的对称破缺态，由于自旋极化，会导致alpha和beta轨道不匹配，此时虽然alpha轨道自己是正交归一的，beta轨道自己也是正交归一的，但是alpha和beta之间不满足正交归一关系。更具体来说，第i号alpha轨道和第i号beta轨道之间重叠积分不为1，这俩轨道既可能轨道形状稍有偏差，也可能完全不同。因此，对于非限制性开壳层波函数，讨论轨道的时候必须分别去考察alpha和beta轨道，这是比较麻烦的事情。而且这种情况，体系的自旋密度是由所有占据轨道所决定的，因此没法只拿某几条轨道来讨论自旋密度。（不知道什么是自旋密度的话看《谈谈自旋密度、自旋布居以及在Multiwfn中的绘制和计算》http://sobereva.com/353）。\n\n虽然限制性开壳层(RO)计算只会产生一套轨道，没有上述U计算的麻烦，但是相对于

In [10]:
docs[0].metadata = "448.md"

In [11]:
docs[0].metadata

'448.md'

In [38]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

markdown_path = "/Users/xiaohuxu/Documents/python/sob_blobs/sobereva_blogs_text/227.md"
loader = UnstructuredMarkdownLoader(markdown_path)

data = loader.load()
# assert len(data) == 1
# assert isinstance(data[0], Document)
readme_content = data[0].page_content
# print(readme_content[:250])
text_splitter = RecursiveCharacterTextSplitter(chunk_size=8000, chunk_overlap=100)
docs = text_splitter.split_documents(data)
# docs[1]

In [39]:
for doc in docs:
    print(f"* {doc.page_content[:100]} [{doc.metadata}]")

* 使用Multiwfn计算激发态间的跃迁偶极矩和各个激发态的偶极矩

URL: http://sobereva.com/227 ID: 227

使用Multiwfn计算激发态间的跃迁偶极矩和各个激发态 [{'source': '/Users/xiaohuxu/Documents/python/sob_blobs/sobereva_blogs_text/227.md'}]
